# 3-1 購買データで「誰が・何を・いつ」買うかを読む / Reading Who Buys What, and When, from Sales Data

『AIに頼んで動かす Python実務データ分析』第3章1節の参照用ノートブックです。
Reference notebook for Chapter 3, Section 1 of *Data Analysis with AI and Python*.

**使い方 / How to use**：最初のセルの `LANG` を `"ja"` または `"en"` にして、すべてのセルを上から実行します。
Set `LANG` in the first cell to `"ja"` or `"en"`, then run all cells from top to bottom.

In [ ]:
# ===== 設定 / Settings =====
LANG = "ja"   # "ja" = 日本語 / "en" = English
SAVE_FIGURES = True
BASE_URL = "https://raw.githubusercontent.com/YOUR_ACCOUNT/YOUR_REPO/main/data/"

In [ ]:
import os, subprocess, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

JP_FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if LANG == "ja":
    if not os.path.exists(JP_FONT):
        subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True, check=True)
    fm.fontManager.addfont(JP_FONT)
    plt.rcParams["font.family"] = fm.FontProperties(fname=JP_FONT).get_name()
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "black", "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 300})
FIG_W = 4.5

def save(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        os.makedirs(f"figures/{LANG}", exist_ok=True)
        fig.savefig(f"figures/{LANG}/{name}.png", bbox_inches="tight")
    plt.show()

In [ ]:
# ===== 表示用のラベル / Display labels =====
L = {
 "ja": dict(sales="売上（万円）", count="顧客数（人）", share="売上構成比（%）", rec="最終購入からの日数（Recency）",
     mon="購買金額の合計（万円）", actual="実際の売上（万円）", pred="予測した売上（万円）", act_line="実際の売上",
     trend="回帰直線（月番号のみ）", m1="月番号のみ", m2="月番号＋注文件数", m3="月番号＋繁忙期",
     pc1="第1主成分（PC1）", pc2="第2主成分（PC2）",
     cats={"Tops": "トップス", "Bottoms": "ボトムス", "Outerwear": "アウター", "Dresses": "ワンピース",
           "Shoes": "シューズ", "Bags": "バッグ", "Accessories": "小物"},
     prods={"T-shirt": "Tシャツ", "Knit Sweater": "ニット", "Blouse": "ブラウス", "Hoodie": "パーカー",
            "Skinny Denim": "スキニーデニム", "Wide Pants": "ワイドパンツ", "Pleated Skirt": "プリーツスカート",
            "Chino Pants": "チノパンツ", "Trench Coat": "トレンチコート", "Down Jacket": "ダウンジャケット",
            "Denim Jacket": "デニムジャケット", "Shirt Dress": "シャツワンピース", "Knit Dress": "ニットワンピース",
            "Sneakers": "スニーカー", "Loafers": "ローファー", "Ankle Boots": "ショートブーツ", "Tote Bag": "トートバッグ",
            "Shoulder Bag": "ショルダーバッグ", "Scarf": "マフラー", "Cap": "キャップ", "Socks Set": "ソックスセット"},
     segs={"Champions": "最優良顧客", "Loyal": "常連顧客", "Potential": "育成顧客", "New Customers": "新規顧客",
           "At Risk": "要注意顧客", "Lost": "離脱顧客"}),
 "en": dict(sales="Sales (¥10,000)", count="Customers", share="Share of sales (%)", rec="Days since last purchase (Recency)",
     mon="Total spend (¥10,000)", actual="Actual sales (¥10,000)", pred="Predicted sales (¥10,000)", act_line="Actual sales",
     trend="Regression line (month only)", m1="Month only", m2="Month + orders", m3="Month + peak season",
     pc1="PC1", pc2="PC2", cats=None, prods=None, segs=None),
}[LANG]
tr = lambda d, k: d[k] if d else k   # 日本語なら置き換え、英語ならそのまま / translate if a mapping exists
U = 10000   # 万円 / ¥10,000

## データの読み込み / Loading the data

In [ ]:
import urllib.request
if not os.path.exists("sales_data.csv"):
    try:
        urllib.request.urlretrieve(BASE_URL + "sales_data.csv", "sales_data.csv")
    except Exception as e:
        print("sales_data.csv をアップロードしてください / Please upload sales_data.csv", e)
df = pd.read_csv("sales_data.csv", parse_dates=["order_date"])
print(df.shape, "欠損 / missing:", int(df.isna().sum().sum()))
print("注文 / orders:", df.order_id.nunique(), " 顧客 / customers:", df.customer_id.nunique(),
      " 売上 / sales: ¥{:,}".format(df.amount.sum()))
print(df.order_date.min().date(), "–", df.order_date.max().date())
df.head()

## 3-1-1 売上を可視化する / Visualizing sales

In [ ]:
m = df.groupby(df.order_date.dt.to_period("M")).agg(sales=("amount", "sum"), orders=("order_id", "nunique")).reset_index()
lab = m.order_date.astype(str)
fig, ax = plt.subplots(figsize=(FIG_W, 2.4))
ax.plot(range(24), m.sales / U, "k-o", ms=3, mfc="white", lw=1)
for i, dy in [(m.sales.idxmax(), 5), (m.sales.idxmin(), -11)]:
    ax.annotate(f"{m.sales[i]/U:.0f}", (i, m.sales[i] / U), xytext=(0, dy), textcoords="offset points", ha="center", fontsize=7)
ax.set_xticks(range(0, 24, 3), lab[::3], rotation=45, ha="right", rotation_mode="anchor", fontsize=7)
ax.set_ylabel(L["sales"]); ax.set_ylim(0, 260)
save(fig, "fig3-1-1_monthly_sales")
print("最高 / max:", lab[m.sales.idxmax()], m.sales.max(), " 最低 / min:", lab[m.sales.idxmin()], m.sales.min())

In [ ]:
c = df.groupby("category").amount.sum().sort_values()
fig, ax = plt.subplots(figsize=(FIG_W, 2.2))
ax.barh([tr(L["cats"], k) for k in c.index], c / U, color=["#999999"] * (len(c) - 1) + ["#333333"], edgecolor="black", lw=0.5)
for i, v in enumerate(c / U): ax.text(v + 5, i, f"{v:,.0f}", va="center", fontsize=7)
ax.set_xlabel(L["sales"]); ax.set_xlim(0, max(c / U) * 1.15)
save(fig, "fig3-1-2_category")
(c / c.sum() * 100).round(1).sort_values(ascending=False)

In [ ]:
p = df.groupby("product_name").amount.sum().sort_values().tail(10)
fig, ax = plt.subplots(figsize=(FIG_W, 2.6))
ax.barh([tr(L["prods"], k) for k in p.index], p / U, color="#777777", edgecolor="black", lw=0.5)
for i, v in enumerate(p / U): ax.text(v + 3, i, f"{v:,.0f}", va="center", fontsize=7)
ax.set_xlabel(L["sales"]); ax.set_xlim(0, max(p / U) * 1.15)
save(fig, "fig3-1-3_top10")

## 3-1-2 RFM分析 / RFM analysis

In [ ]:
ref = pd.Timestamp("2026-01-01")   # 基準日 / reference date
r = df.groupby("customer_id").agg(last=("order_date", "max"), F=("order_id", "nunique"), M=("amount", "sum"))
r["R"] = (ref - r["last"]).dt.days
r["R_score"] = pd.qcut(r.R.rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
r["F_score"] = pd.qcut(r.F.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
r["M_score"] = pd.qcut(r.M.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment(x):
    if x.R_score >= 4 and x.F_score >= 4: return "Champions"
    if x.R_score >= 4 and x.F_score == 1: return "New Customers"
    if x.R_score >= 3 and x.F_score >= 3: return "Loyal"
    if x.R_score >= 3: return "Potential"
    if x.F_score >= 3: return "At Risk"
    return "Lost"
r["segment"] = r.apply(segment, axis=1)
ORDER = ["Champions", "Loyal", "Potential", "New Customers", "At Risk", "Lost"]
summary = r.groupby("segment").agg(n=("R", "size"), R=("R", "mean"), F=("F", "mean"), M=("M", "mean")).reindex(ORDER).round(0)
summary["share_%"] = (r.groupby("segment").M.sum() / r.M.sum() * 100).reindex(ORDER).round(1)
summary

In [ ]:
cnt = summary["n"]; sh = summary["share_%"]
yl = [tr(L["segs"], s) for s in ORDER][::-1]
fig, axs = plt.subplots(1, 2, figsize=(FIG_W, 2.4), sharey=True)
axs[0].barh(yl, cnt.values[::-1], color="#bbbbbb", edgecolor="black", lw=0.5)
axs[1].barh(yl, sh.values[::-1], color="#555555", edgecolor="black", lw=0.5)
for i, v in enumerate(cnt.values[::-1]): axs[0].text(v + 1, i, f"{v:.0f}", va="center", fontsize=7)
for i, v in enumerate(sh.values[::-1]): axs[1].text(v + 1, i, f"{v:.0f}", va="center", fontsize=7)
axs[0].set_xlabel(L["count"]); axs[1].set_xlabel(L["share"]); axs[0].set_xlim(0, 52); axs[1].set_xlim(0, 68)
save(fig, "fig3-1-4_segments")

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(FIG_W, 3.2), sharex=True, sharey=True)
for ax, s in zip(axs.flat, ORDER):
    ax.scatter(r[r.segment != s].R, r[r.segment != s].M / U, s=5, color="#cccccc", lw=0)
    ax.scatter(r[r.segment == s].R, r[r.segment == s].M / U, s=9, color="black", lw=0)
    ax.set_title(tr(L["segs"], s), fontsize=7.5); ax.tick_params(labelsize=6.5)
fig.supxlabel(L["rec"], fontsize=7.5); fig.supylabel(L["mon"], fontsize=7.5)
save(fig, "fig3-1-5_rfm_scatter")

## 3-1-3 売上を予測する / Forecasting sales

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
m["t"] = np.arange(1, 25)
m["peak"] = m.order_date.dt.month.isin([10, 11, 12]).astype(int)

models = {L["m1"]: ["t"], L["m2"]: ["t", "orders"], L["m3"]: ["t", "peak"]}
rows, preds = [], {}
for name, cols in models.items():
    lr = LinearRegression().fit(m[cols], m.sales); pr = lr.predict(m[cols]); preds[name] = pr
    rows.append([name, dict(zip(cols, lr.coef_.round(0))), round(lr.intercept_), round(r2_score(m.sales, pr), 3),
                 round(mean_squared_error(m.sales, pr) ** 0.5)])
pd.DataFrame(rows, columns=["model", "coef", "intercept", "R2", "RMSE"])

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W, 2.4))
ax.plot(range(24), m.sales / U, "k-o", ms=3, mfc="white", lw=1, label=L["act_line"])
ax.plot(range(24), preds[L["m1"]] / U, "k--", lw=1.2, label=L["trend"])
ax.set_xticks(range(0, 24, 3), lab[::3], rotation=45, ha="right", rotation_mode="anchor", fontsize=7)
ax.set_ylabel(L["sales"]); ax.set_ylim(0, 260); ax.legend(fontsize=7, frameon=False, loc="upper left")
save(fig, "fig3-1-6_trend")

fig, axs = plt.subplots(1, 3, figsize=(FIG_W, 1.9), sharex=True, sharey=True)
for ax, (name, pr) in zip(axs, preds.items()):
    ax.plot([0, 260], [0, 260], "k--", lw=0.7)
    ax.scatter(m.sales / U, pr / U, s=10, facecolor="white", edgecolor="black", lw=0.6)
    ax.set_title(f"{name}\nR² = {r2_score(m.sales, pr):.2f}", fontsize=7); ax.tick_params(labelsize=6.5)
    ax.set_aspect("equal"); ax.set_xlim(0, 260); ax.set_ylim(0, 260)
fig.supxlabel(L["actual"], fontsize=7.5); axs[0].set_ylabel(L["pred"], fontsize=7.5)
save(fig, "fig3-1-7_pred_vs_actual")

## 3-1-4 主成分分析 / Principal component analysis

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Z = StandardScaler().fit_transform(r[["R", "F", "M"]])
pca = PCA(2).fit(Z)
# 符号を本文の表にそろえる（F が PC1 でプラス、R が PC2 でプラス） / Align signs with the book
sgn = np.array([np.sign(pca.components_[0, 1]), np.sign(pca.components_[1, 0])])
comp = pca.components_ * sgn[:, None]
S = pca.transform(Z) * sgn
r["PC1"], r["PC2"] = S[:, 0], S[:, 1]
ev = pca.explained_variance_ratio_ * 100
print("寄与率 / explained variance (%):", ev.round(1), " 累積 / cumulative:", ev.sum().round(1))
print("相関 F–M / correlation:", round(r[["F", "M"]].corr().iloc[0, 1], 2))
pd.DataFrame(comp.T, index=["Recency", "Frequency", "Monetary"], columns=["PC1", "PC2"]).round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W, 3.2))
ax.scatter(r.PC1, r.PC2, s=6, color="#aaaaaa", lw=0)
loff = {"New Customers": (-6, -3, "right"), "Potential": (6, -3, "left")}
for s, (a, b) in r.groupby("segment")[["PC1", "PC2"]].mean().iterrows():
    dx, dy, ha = loff.get(s, (5, 4, "left"))
    ax.scatter(a, b, marker="s", s=28, color="black")
    ax.annotate(tr(L["segs"], s), (a, b), xytext=(dx, dy), textcoords="offset points", fontsize=7, fontweight="bold", ha=ha)
K = 1.8
for i, (v, (ox, oy, ha)) in enumerate(zip(["Recency", "Frequency", "Monetary"],
                                          [(-0.05, 0.08, "right"), (0.08, -0.12, "left"), (0.02, 0.1, "left")])):
    x, y = comp[0, i] * K, comp[1, i] * K
    ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", lw=1, color="black", mutation_scale=8))
    ax.text(x + ox, y + oy, v, fontsize=7, ha=ha, style="italic")
ax.axhline(0, color="gray", lw=0.5, ls=":"); ax.axvline(0, color="gray", lw=0.5, ls=":")
ax.set_ylim(-1.7, 2.0); ax.set_xlabel(f"{L['pc1']} {ev[0]:.0f}%"); ax.set_ylabel(f"{L['pc2']} {ev[1]:.0f}%")
save(fig, "fig3-1-8_biplot")